# 🌲 YTÜ Harita Mühendisliği - Çam Ağacı Instance Segmentation Eğitimi

**Tez Konusu:** İHA görüntülerinden makine öğrenmesi ile çam ağaçlarının otomatik tespiti ve segmentasyonu

**Model:** YOLOv8m-seg (COCO pretrained → transfer learning)

**Veri:** Roboflow'dan YOLOv8-seg formatında

---

## 1. GPU Kontrolü ve Kurulum

Google Colab'da T4 GPU'nun aktif olduğundan emin olun:
- **Runtime → Change runtime type → GPU (T4)**

In [ ]:
# GPU kontrolü - T4 veya daha iyi bir GPU görünmeli
!nvidia-smi

import torch
print(f"\n✅ PyTorch sürümü: {torch.__version__}")
print(f"✅ CUDA kullanılabilir: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ UYARI: GPU bulunamadı! Runtime → Change runtime type → GPU seçin.")

In [ ]:
# Gerekli paketlerin kurulumu
# ultralytics: YOLOv8 framework'ü
# roboflow: Veri seti indirme

!pip install ultralytics==8.3.0 roboflow==1.1.46 -q

print("✅ Paketler başarıyla kuruldu!")

## 2. Roboflow'dan Veri Seti İndirme

Roboflow'da etiketlediğiniz **cam-segmentation** projesini YOLOv8-seg formatında indiriyoruz.

⚠️ **API anahtarınızı aşağıya girin** (Roboflow → Settings → API Key)

In [ ]:
#==============================================================================
# KONFİGÜRASYON - Bu değerleri kendi projenize göre düzenleyin
#==============================================================================

ROBOFLOW_API_KEY = "BURAYA_API_ANAHTARINIZI_GIRIN"  # Roboflow API anahtarı
ROBOFLOW_WORKSPACE = "BURAYA_WORKSPACE_ADINIZI_GIRIN"  # Workspace adı (URL'den bakın)
ROBOFLOW_PROJECT = "cam-segmentation"  # Proje adı
ROBOFLOW_VERSION = 1  # Dataset versiyonu

#==============================================================================

In [ ]:
from roboflow import Roboflow
import os

# API anahtarı kontrolü
if "BURAYA" in ROBOFLOW_API_KEY:
    raise ValueError("❌ HATA: Lütfen ROBOFLOW_API_KEY değişkenine kendi API anahtarınızı girin!")

print("📥 Roboflow'dan veri seti indiriliyor...")
print(f"   Workspace: {ROBOFLOW_WORKSPACE}")
print(f"   Proje: {ROBOFLOW_PROJECT}")
print(f"   Versiyon: {ROBOFLOW_VERSION}")

# Roboflow'a bağlan ve veri setini indir
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("yolov8-seg")

# Veri seti yolu
DATASET_PATH = dataset.location
print(f"\n✅ Veri seti indirildi: {DATASET_PATH}")

# Veri seti yapısını kontrol et
print("\n📁 Veri seti yapısı:")
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # Sadece 2 seviye derinliğe kadar göster
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # İlk 5 dosyayı göster
            print(f"{subindent}{file}")
        if len(files) > 5:
            print(f"{subindent}... ve {len(files)-5} dosya daha")

In [ ]:
# Eğitim ve doğrulama görüntü sayılarını kontrol et
import glob

train_images = glob.glob(os.path.join(DATASET_PATH, "train", "images", "*"))
val_images = glob.glob(os.path.join(DATASET_PATH, "valid", "images", "*"))

print(f"📊 Veri Seti İstatistikleri:")
print(f"   Eğitim görselleri: {len(train_images)}")
print(f"   Doğrulama görselleri: {len(val_images)}")
print(f"   Toplam: {len(train_images) + len(val_images)}")
print(f"   Train/Val oranı: %{100*len(train_images)//(len(train_images)+len(val_images))} / %{100*len(val_images)//(len(train_images)+len(val_images))}")

if len(train_images) < 30:
    print("\n⚠️ UYARI: Eğitim verisi az (<30). Daha fazla etiketleme yapmanız önerilir.")

## 3. Örnek Görüntüleri Görselleştirme

Eğitime başlamadan önce veri setinden birkaç örnek görelim.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random

# Rastgele 4 eğitim görüntüsü seç ve göster
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
fig.suptitle("📸 Eğitim Veri Setinden Örnekler", fontsize=14, fontweight='bold')

sample_images = random.sample(train_images, min(4, len(train_images)))

for ax, img_path in zip(axes.flatten(), sample_images):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path), fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("💡 İpucu: Etiketler images klasörünün yanındaki labels klasöründe .txt olarak saklanıyor.")

## 4. YOLOv8m-seg Modelini Yükle

COCO veri seti üzerinde önceden eğitilmiş **YOLOv8m-seg** modelini yüklüyoruz.

- **m (medium)**: Hız ve doğruluk dengesi için ideal
- **seg**: Instance segmentation versiyonu (bounding box + mask)

In [ ]:
from ultralytics import YOLO

# YOLOv8m-seg modelini indir ve yükle (COCO pretrained)
print("📦 YOLOv8m-seg modeli yükleniyor (COCO pretrained)...")
model = YOLO("yolov8m-seg.pt")

print("\n✅ Model başarıyla yüklendi!")
print(f"   Model tipi: {model.task}")
print(f"   Parametre sayısı: ~25.3M")
print("\n📋 Model mimarisi özeti:")
print("   - Backbone: CSPDarknet (özellik çıkarımı)")
print("   - Neck: PANet (çok ölçekli özellik füzyonu)")
print("   - Head: Decoupled head (sınıf + bbox + mask)")

## 5. Model Eğitimi

### Eğitim Parametreleri:
| Parametre | Değer | Açıklama |
|-----------|-------|----------|
| `epochs` | 150 | Maksimum eğitim turu |
| `patience` | 30 | Early stopping için sabır (30 epoch iyileşme olmazsa dur) |
| `imgsz` | 640 | Giriş görüntü boyutu (640×640 px) |
| `batch` | -1 (auto) | GPU belleğine göre otomatik batch size |
| `optimizer` | auto | AdamW veya SGD (otomatik seçim) |
| `lr0` | 0.01 | Başlangıç öğrenme hızı |
| `lrf` | 0.01 | Son öğrenme hızı (lr0 × lrf) |

### Data Augmentation (Varsayılan - Aktif):
- **Mosaic (p=1.0)**: 4 görüntüyü birleştirir, küçük nesneleri öğrenmeyi iyileştirir
- **Flip horizontal (p=0.5)**: Yatay çevirme
- **HSV augmentation**: Renk, doygunluk, parlaklık değişimi
- **Scale (±50%)**: Ölçek değişimi

In [ ]:
#==============================================================================
# EĞİTİM PARAMETRELERİ
#==============================================================================

EPOCHS = 150        # Maksimum epoch sayısı
PATIENCE = 30       # Early stopping patience
IMAGE_SIZE = 640    # Giriş boyutu (640x640)
BATCH_SIZE = -1     # Otomatik (GPU belleğine göre)
SEED = 42           # Rastgelelik tohumu (tekrarlanabilirlik, tezde belirtilecek)

#==============================================================================

In [ ]:
# data.yaml dosyasının yolunu bul
data_yaml = os.path.join(DATASET_PATH, "data.yaml")

if not os.path.exists(data_yaml):
    raise FileNotFoundError(f"❌ HATA: data.yaml bulunamadı: {data_yaml}")

print(f"📄 data.yaml yolu: {data_yaml}")

# data.yaml içeriğini göster
print("\n📋 data.yaml içeriği:")
print("-" * 40)
with open(data_yaml, 'r') as f:
    print(f.read())
print("-" * 40)

In [ ]:
%%time

# MODEL EĞİTİMİ BAŞLIYOR
print("="*60)
print("🚀 YOLOv8m-seg EĞİTİMİ BAŞLIYOR")
print("="*60)
print(f"   Epochs: {EPOCHS}")
print(f"   Patience: {PATIENCE}")
print(f"   Image Size: {IMAGE_SIZE}")
print(f"   Batch Size: {'Otomatik' if BATCH_SIZE == -1 else BATCH_SIZE}")
print(f"   Seed: {SEED} (deterministic=True)")
print("="*60)

# Eğitimi başlat
results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    
    # Optimizasyon
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    
    # Data Augmentation (varsayılanlar iyi çalışır)
    hsv_h=0.015,      # Hue değişimi
    hsv_s=0.7,        # Saturation değişimi
    hsv_v=0.4,        # Value (parlaklık) değişimi
    degrees=0.0,      # Rotasyon (ağaçlar için kapalı)
    translate=0.1,    # Öteleme
    scale=0.5,        # Ölçekleme (±50%)
    flipud=0.0,       # Dikey çevirme (kapalı - ağaçlar ters durmaz)
    fliplr=0.5,       # Yatay çevirme
    mosaic=1.0,       # Mosaic augmentation
    mixup=0.0,        # Mixup (kapalı)
    
    # Tekrarlanabilirlik (tezde belirtilecek: seed=42, deterministic=True)
    seed=SEED,
    deterministic=True,
    
    # Diğer ayarlar
    device=0,         # GPU 0
    workers=2,        # Colab için 2 worker yeterli
    project="cam_segmentation",
    name="yolov8m_seg_egitim",
    exist_ok=True,
    pretrained=True,
    verbose=True,
    save=True,
    save_period=25,   # Her 25 epoch'ta checkpoint kaydet
    plots=True,       # Eğitim grafiklerini oluştur
)

print("\n" + "="*60)
print("✅ EĞİTİM TAMAMLANDI!")
print("="*60)

## 6. Eğitim Sonuçlarını Değerlendirme

Eğitim bittikten sonra model performansını değerlendiriyoruz.

In [ ]:
# Eğitim çıktı klasörünü bul
TRAIN_DIR = os.path.join("cam_segmentation", "yolov8m_seg_egitim")

# En iyi model ağırlıklarının yolu
BEST_MODEL_PATH = os.path.join(TRAIN_DIR, "weights", "best.pt")
LAST_MODEL_PATH = os.path.join(TRAIN_DIR, "weights", "last.pt")

print(f"📁 Eğitim çıktı klasörü: {TRAIN_DIR}")
print(f"🏆 En iyi model: {BEST_MODEL_PATH}")
print(f"📄 Son model: {LAST_MODEL_PATH}")

# Dosyaların varlığını kontrol et
if os.path.exists(BEST_MODEL_PATH):
    size_mb = os.path.getsize(BEST_MODEL_PATH) / (1024*1024)
    print(f"\n✅ best.pt bulundu ({size_mb:.1f} MB)")
else:
    print("\n❌ UYARI: best.pt bulunamadı!")

In [ ]:
# Eğitim grafiklerini göster
from IPython.display import Image, display

print("📊 EĞİTİM GRAFİKLERİ")
print("=" * 50)

# results.png - Tüm eğitim metrikleri
results_png = os.path.join(TRAIN_DIR, "results.png")
if os.path.exists(results_png):
    print("\n📈 Loss ve Metrik Eğrileri:")
    display(Image(filename=results_png, width=900))
else:
    print("⚠️ results.png bulunamadı")

In [ ]:
# Confusion Matrix
confusion_matrix_png = os.path.join(TRAIN_DIR, "confusion_matrix.png")
confusion_matrix_norm_png = os.path.join(TRAIN_DIR, "confusion_matrix_normalized.png")

print("\n🎯 CONFUSION MATRIX (Karışıklık Matrisi)")
print("-" * 50)

if os.path.exists(confusion_matrix_norm_png):
    display(Image(filename=confusion_matrix_norm_png, width=600))
elif os.path.exists(confusion_matrix_png):
    display(Image(filename=confusion_matrix_png, width=600))
else:
    print("⚠️ Confusion matrix bulunamadı")

In [ ]:
# PR Curve (Precision-Recall)
print("\n📉 PRECISION-RECALL EĞRİSİ")
print("-" * 50)

# Mask PR curve
pr_curve_mask = os.path.join(TRAIN_DIR, "MaskPR_curve.png")
pr_curve_box = os.path.join(TRAIN_DIR, "BoxPR_curve.png")

if os.path.exists(pr_curve_mask):
    print("Mask Segmentation PR Curve:")
    display(Image(filename=pr_curve_mask, width=600))

if os.path.exists(pr_curve_box):
    print("\nBounding Box PR Curve:")
    display(Image(filename=pr_curve_box, width=600))

In [ ]:
# Doğrulama seti üzerinde değerlendirme
print("\n📊 DOĞRULAMA SETİ DEĞERLENDİRMESİ")
print("=" * 50)

# En iyi modeli yükle ve değerlendir
best_model = YOLO(BEST_MODEL_PATH)
metrics = best_model.val(data=data_yaml, imgsz=IMAGE_SIZE, split="val")

print("\n" + "=" * 50)
print("📈 SONUÇ METRİKLERİ")
print("=" * 50)

# Box metrikleri
print("\n🔲 BOUNDING BOX METRİKLERİ:")
print(f"   mAP50 (Box):     {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)")
print(f"   mAP50-95 (Box):  {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)")

# Mask metrikleri
print("\n🎭 MASK (SEGMENTATION) METRİKLERİ:")
print(f"   mAP50 (Mask):    {metrics.seg.map50:.4f}  ({metrics.seg.map50*100:.1f}%)")
print(f"   mAP50-95 (Mask): {metrics.seg.map:.4f}  ({metrics.seg.map*100:.1f}%)")

print("\n" + "=" * 50)
print("💡 Yorumlama:")
print("   - mAP50 > 0.7: İyi performans")
print("   - mAP50 > 0.85: Çok iyi performans")
print("   - mAP50-95: Daha katı metrik (0.5'ten 0.95'e IoU eşikleri)")
print("=" * 50)

In [ ]:
# Örnek tahmin görselleştirmesi
print("\n🖼️ ÖRNEK TAHMİN GÖRSELLERİ")
print("-" * 50)

# Doğrulama setinden rastgele görüntüler üzerinde tahmin
val_batch_png = os.path.join(TRAIN_DIR, "val_batch0_pred.png")
val_batch_labels_png = os.path.join(TRAIN_DIR, "val_batch0_labels.png")

if os.path.exists(val_batch_labels_png):
    print("Gerçek Etiketler (Ground Truth):")
    display(Image(filename=val_batch_labels_png, width=800))

if os.path.exists(val_batch_png):
    print("\nModel Tahminleri (Predictions):")
    display(Image(filename=val_batch_png, width=800))

## 7. Modeli Google Drive'a Kaydet

Eğitilmiş modeli (best.pt) Google Drive'a kaydederek kalıcı hale getiriyoruz.

In [ ]:
# Google Drive'ı bağla
from google.colab import drive

print("📁 Google Drive bağlanıyor...")
drive.mount('/content/drive')
print("✅ Google Drive bağlandı!")

In [ ]:
import shutil

# Kayıt klasörü oluştur
DRIVE_SAVE_DIR = "/content/drive/MyDrive/YTU_Tez_Cam_Segmentation"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# best.pt'yi kopyala
drive_best_path = os.path.join(DRIVE_SAVE_DIR, "best.pt")
shutil.copy2(BEST_MODEL_PATH, drive_best_path)
print(f"✅ best.pt kaydedildi: {drive_best_path}")

# Son modeli de kopyala (yedek olarak)
if os.path.exists(LAST_MODEL_PATH):
    drive_last_path = os.path.join(DRIVE_SAVE_DIR, "last.pt")
    shutil.copy2(LAST_MODEL_PATH, drive_last_path)
    print(f"✅ last.pt kaydedildi: {drive_last_path}")

# Eğitim grafiklerini de kopyala
if os.path.exists(results_png):
    shutil.copy2(results_png, os.path.join(DRIVE_SAVE_DIR, "results.png"))
    print(f"✅ results.png kaydedildi")

print(f"\n📂 Tüm dosyalar kaydedildi: {DRIVE_SAVE_DIR}")
print("\n💡 İpucu: best.pt dosyasını Windows bilgisayarınıza indirip inference için kullanın.")

## 8. Test: Tek Görüntü Üzerinde Tahmin

Modelin doğru çalıştığını test etmek için tek bir görüntü üzerinde tahmin yapalım.

In [ ]:
# Doğrulama setinden rastgele bir görüntü seç
test_image = random.choice(val_images)
print(f"🖼️ Test görüntüsü: {test_image}")

# Tahmin yap
results = best_model.predict(
    source=test_image,
    conf=0.4,           # Güven eşiği
    iou=0.5,            # NMS IoU eşiği
    imgsz=IMAGE_SIZE,
    save=True,
    save_txt=False,
    project="test_predictions",
    name="single_test",
    exist_ok=True,
)

# Sonuçları göster
result = results[0]

print(f"\n📊 Tahmin Sonuçları:")
print(f"   Tespit edilen nesne sayısı: {len(result.boxes)}")

if len(result.boxes) > 0:
    print(f"   Güven skorları: {[f'{c:.2f}' for c in result.boxes.conf.tolist()]}")

# Görselleştirme
print("\n🖼️ Tahmin Görseli:")
pred_img_path = os.path.join("test_predictions", "single_test", os.path.basename(test_image))
if os.path.exists(pred_img_path):
    display(Image(filename=pred_img_path, width=800))
else:
    # Alternatif görselleştirme
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(result.plot())
    ax.axis('off')
    ax.set_title(f"Tespit: {len(result.boxes)} çam ağacı")
    plt.tight_layout()
    plt.show()

## 9. Sonraki Adımlar

Eğitim tamamlandı! Şimdi yapmanız gerekenler:

1. **best.pt dosyasını indirin** (Google Drive'dan)
2. **Windows bilgisayarınıza kopyalayın**
3. **inference_orthomosaic.py** scriptini çalıştırın (ortomozaik üzerinde)
4. **export_results.py** ile CBS çıktılarını oluşturun

---

**Model Özeti:**
- Model: YOLOv8m-seg
- Sınıf: cam (çam ağacı)
- Çıktı: Bounding box + Instance mask
- Format: .pt (PyTorch)

In [ ]:
# Final özet
print("="*60)
print("🎉 EĞİTİM BAŞARIYLA TAMAMLANDI!")
print("="*60)
print(f"\n📁 Model konumu: {drive_best_path}")
print(f"📊 mAP50 (Mask): {metrics.seg.map50*100:.1f}%")
print(f"📊 mAP50-95 (Mask): {metrics.seg.map*100:.1f}%")
print("\n📋 Sonraki adımlar:")
print("   1. best.pt dosyasını Drive'dan indirin")
print("   2. inference_orthomosaic.py ile ortomozaiği işleyin")
print("   3. export_results.py ile shapefile oluşturun")
print("   4. QGIS'te sonuçları görselleştirin")
print("="*60)

## 10. Eğitim Metrikleri Tablosu (Tez İçin)

Aşağıdaki hücre, doğrulama metriklerini (mAP50, mAP50-95, Precision, Recall — Box ve Mask ayrı)
tez tablosuna hazır bir pandas DataFrame olarak yazdırır ve `egitim_metrikleri.csv` dosyasına kaydeder.
Google Drive bağlıysa CSV, model klasörüne de kopyalanır.

In [ ]:
# Eğitim metriklerini tez tablosuna hazır DataFrame olarak yazdır ve CSV'ye kaydet
# NOT: Bu hücre, 6. bölümdeki değerlendirme hücresinin (metrics = best_model.val(...))
# çalıştırılmış olmasını gerektirir.

import pandas as pd

metrik_tablosu = pd.DataFrame({
    "Metrik": ["mAP50", "mAP50-95", "Precision", "Recall"],
    "Box (Sınırlayıcı Kutu)": [
        round(metrics.box.map50, 4),
        round(metrics.box.map, 4),
        round(metrics.box.mp, 4),
        round(metrics.box.mr, 4),
    ],
    "Mask (Segmentasyon)": [
        round(metrics.seg.map50, 4),
        round(metrics.seg.map, 4),
        round(metrics.seg.mp, 4),
        round(metrics.seg.mr, 4),
    ],
})

print("📊 EĞİTİM METRİKLERİ TABLOSU (TEZ İÇİN)")
print("=" * 50)
print(metrik_tablosu.to_string(index=False))
print("=" * 50)

# CSV olarak kaydet (utf-8-sig: Excel'de Türkçe karakterler düzgün görünür)
csv_yolu = "egitim_metrikleri.csv"
metrik_tablosu.to_csv(csv_yolu, index=False, encoding="utf-8-sig")
print(f"\n✅ CSV kaydedildi: {csv_yolu}")

# Google Drive bağlıysa model klasörüne de kopyala
import os, shutil
if os.path.isdir("/content/drive/MyDrive"):
    drive_klasor = "/content/drive/MyDrive/YTU_Tez_Cam_Segmentation"
    os.makedirs(drive_klasor, exist_ok=True)
    shutil.copy2(csv_yolu, os.path.join(drive_klasor, "egitim_metrikleri.csv"))
    print(f"✅ Drive'a kopyalandı: {drive_klasor}/egitim_metrikleri.csv")
else:
    print("💡 Google Drive bağlı değil; CSV sadece Colab çalışma alanına kaydedildi.")

# Tablonun şık görünümü (notebook çıktısı)
metrik_tablosu
